In [2]:
import json
from pathlib import Path

chunks_path = Path(r"C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\data\tinystories_chunks.json")

with chunks_path.open("r", encoding="utf-8") as file:
    chunks = json.load(file)

print(type(chunks))
print(f"chunk 数量：{len(chunks)}")
print(chunks[0])

<class 'list'>
chunk 数量：10417
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other


In [4]:
import json
import shutil
from pathlib import Path

chunks_path = Path(r"C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\data\tinystories_chunks.json")
backup_path = chunks_path.with_suffix(".json.bak")
temp_path = chunks_path.with_suffix(".json.tmp")

with chunks_path.open("r", encoding="utf-8") as file:
    raw_chunks = json.load(file)

assert isinstance(raw_chunks, list)
assert raw_chunks
assert all(isinstance(text, str) and text.strip() for text in raw_chunks)

structured_chunks = [
    {
        "chunk_id": str(position),
        "text": text,
        "source": "tinystories",
        "metadata": {
            "faiss_id": position,
        },
    }
    for position, text in enumerate(raw_chunks)
]

# 保留一份原始文件，确认无误后可以自行删除
shutil.copy2(chunks_path, backup_path)

# 先写临时文件，成功后再替换原文件
with temp_path.open("w", encoding="utf-8") as file:
    json.dump(
        structured_chunks,
        file,
        ensure_ascii=False,
        indent=2,
    )

temp_path.replace(chunks_path)

print(f"Migrated {len(structured_chunks)} chunks")
print(f"Backup: {backup_path}")

Migrated 10417 chunks
Backup: C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\data\tinystories_chunks.json.bak


In [5]:
with chunks_path.open("r", encoding="utf-8") as file:
    chunks = json.load(file)

assert len(chunks) == len(raw_chunks)
assert chunks[0]["chunk_id"] == "0"
assert chunks[0]["text"] == raw_chunks[0]
assert chunks[-1]["metadata"]["faiss_id"] == len(chunks) - 1

print(chunks[0])

{'chunk_id': '0', 'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other', 'source': 'tinystories', 'metadata': {'faiss_id': 0}}


In [13]:
import faiss

index_path = r"C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\data\tinystories.faiss"
index = faiss.read_index(str(index_path))

print("FAISS dimension:", index.d)
print("FAISS vectors:", index.ntotal)
assert index.ntotal == len(chunks)


FAISS dimension: 384
FAISS vectors: 10417


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system"
)
SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)
print("Python:", sys.executable)

Project root: C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system
Source directory: C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\src
Python: C:\Users\admin\anaconda3\envs\hf_llm\python.exe


In [2]:
from rag_agent_system.config import load_config
from rag_agent_system.ingestion import load_chunks

config = load_config()
chunk_store = load_chunks(config.paths.chunks)

print("FAISS index:", config.paths.faiss_index)
print("Chunk store:", config.paths.chunks)
print("Chunk count:", len(chunk_store))

FAISS index: C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\data\tinystories.faiss
Chunk store: C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\data\tinystories_chunks.json
Chunk count: 10417


In [11]:
import importlib
import importlib.util
import sys
from pathlib import Path

module_path = Path(
    r"C:\Users\admin\PycharmProjects\Deeplearning_Study"
    r"\rag_agent_system\src\rag_agent_system\config.py"
)

pyc_path = Path(
    importlib.util.cache_from_source(str(module_path))
)

print("Source:", module_path)
print("Cached bytecode:", pyc_path)

# 删除旧的编译缓存；Python 会自动重新生成
pyc_path.unlink(missing_ok=True)

# 移除内存中的旧模块
sys.modules.pop("rag_agent_system.config", None)

importlib.invalidate_caches()

# 重新导入，必须用返回的新模块对象
config_module = importlib.import_module(
    "rag_agent_system.config"
)

print("Reloaded from:", config_module.__file__)

Source: C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\src\rag_agent_system\config.py
Cached bytecode: C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\src\rag_agent_system\__pycache__\config.cpython-311.pyc
Reloaded from: C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\src\rag_agent_system\config.py


In [6]:
import dis

dis.dis(config_module.load_config)

 48           0 RESUME                   0

 51           2 LOAD_GLOBAL              1 (NULL + project_root)
             14 PRECALL                  0
             18 CALL                     0
             28 STORE_FAST               1 (root)

 52          30 LOAD_GLOBAL              3 (NULL + resolve_project_path)
             42 LOAD_FAST                0 (path)
             44 JUMP_IF_TRUE_OR_POP      1 (to 48)
             46 LOAD_CONST               1 ('configs/default.yaml')
        >>   48 LOAD_FAST                1 (root)
             50 PRECALL                  2
             54 CALL                     2
             64 STORE_FAST               2 (config_path)

 53          66 LOAD_FAST                2 (config_path)
             68 LOAD_METHOD              2 (is_file)
             90 PRECALL                  0
             94 CALL                     0
            104 POP_JUMP_FORWARD_IF_TRUE    18 (to 142)

 54         106 LOAD_GLOBAL              7 (NULL + FileNotFoundEr

In [7]:
config = config_module.load_config()

print(config.retrieval.embedding_model)
print(config.retrieval.normalize_embeddings)
print(type(config.retrieval.normalize_embeddings))

sentence-transformers/all-MiniLM-L6-v2
True
<class 'bool'>


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system"
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [3]:
import inspect
import rag_agent_system.config as config_module

print("Loaded from:")
print(config_module.__file__)

print("\nLive load_config source:")
print(inspect.getsource(config_module.load_config))

Loaded from:
C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\src\rag_agent_system\config.py

Live load_config source:
def load_config(path: str | Path | None = None) -> AppConfig:
    """Load and minimally validate project configuration from YAML."""

    root = project_root()
    config_path = resolve_project_path(path or "configs/default.yaml", root)
    if not config_path.is_file():
        raise FileNotFoundError(f"Configuration file not found: {config_path}")

    with config_path.open("r", encoding="utf-8") as handle:
        raw = yaml.safe_load(handle) or {}
    if not isinstance(raw, dict):
        raise ValueError("Configuration root must be a mapping.")

    project = _section(raw, "project")
    paths = _section(raw, "paths")
    retrieval = _section(raw, "retrieval")
    generation = _section(raw, "generation")

    top_k = int(retrieval.get("top_k", 3))

    if top_k < 1:
        raise ValueError(
            "retrieval.top_k must be at least 1."
      

In [13]:
from pathlib import Path

source_path = Path(
    r"C:\Users\admin\PycharmProjects\Deeplearning_Study"
    r"\rag_agent_system\src\rag_agent_system"
    r"\retrieval\faiss_retriever.py"
)

lines = source_path.read_text(
    encoding="utf-8"
).splitlines()

for number, line in enumerate(lines, start=1):
    if 25 <= number <= 45:
        leading_length = len(line) - len(line.lstrip())
        leading = line[:leading_length]

        print(
            number,
            f"indent={leading_length}",
            repr(leading),
            repr(line.lstrip()),
        )

25 indent=8 '        ' 'self.normalize_embeddings = normalize_embeddings'
26 indent=0 '' ''
27 indent=8 '        ' 'self._index: faiss.Index | None = None'
28 indent=8 '        ' 'self._chunks: dict[str, DocumentChunk] | None = None'
29 indent=8 '        ' 'self._encoder: SentenceTransformer | None = None'
30 indent=0 '' ''
31 indent=4 '    ' 'def retrieve('
32 indent=12 '            ' 'self,'
33 indent=12 '            ' 'query: str,'
34 indent=12 '            ' 'top_k: int = 3,'
35 indent=4 '    ' ') -> list[RetrievedChunk]:'
36 indent=8 '        ' '"""Return chunks ranked by vector relevance."""'
37 indent=0 '' ''
38 indent=8 '        ' 'raise NotImplementedError('
39 indent=12 '            ' '"TODO(DAY18): implement FaissRetriever.retrieve"'
40 indent=8 '        ' ')'


In [15]:
import importlib
import importlib.util
import sys

pyc_path = Path(
    importlib.util.cache_from_source(
        str(source_path)
    )
)

print("Removing:", pyc_path)
pyc_path.unlink(missing_ok=True)

sys.modules.pop(
    "rag_agent_system.retrieval.faiss_retriever",
    None,
)
sys.modules.pop(
    "rag_agent_system.retrieval",
    None,
)

importlib.invalidate_caches()

Removing: C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\src\rag_agent_system\retrieval\__pycache__\faiss_retriever.cpython-311.pyc


In [16]:
retrieval_module = importlib.import_module(
    "rag_agent_system.retrieval"
)

FaissRetriever = retrieval_module.FaissRetriever

print(FaissRetriever)

W0930 11:18:17.513000 13800 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


<class 'rag_agent_system.retrieval.faiss_retriever.FaissRetriever'>


In [17]:
from rag_agent_system.config import load_config

config = load_config()

retriever = FaissRetriever(
    index_path=config.paths.faiss_index,
    chunks_path=config.paths.chunks,
    embedding_model=config.retrieval.embedding_model,
    normalize_embeddings=(
        config.retrieval.normalize_embeddings
    ),
)

print(retriever.index_path)
print(retriever.chunks_path)
print(retriever.embedding_model)
print(retriever.normalize_embeddings)

C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\data\tinystories.faiss
C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\data\tinystories_chunks.json
sentence-transformers/all-MiniLM-L6-v2
True
